# 07 Check-in Updates: Exogenous Design and Model Upgrades

This notebook audits and deepens the latest model improvements for PHP FX forecasting.

Scope:
- Inspect persisted parameter tables from steps 03 and 04
- Check high-impact exogenous-variable coverage
- Engineer practical proxy features (policy shock, remittance seasonality)
- Validate stationarity alignment (returns vs levels)
- Diagnose multicollinearity via VIF
- Run lead-lag cross-correlation checks
- Test structural-break slope interactions

In [42]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.outliers_influence import variance_inflation_factor

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')
from statistical_validation import load_config, get_project_paths

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 300)

In [43]:
config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
paths = get_project_paths(config)

processed_dir = paths['processed_dir']
results_dir = paths['results_dir']
eval_dir = results_dir / 'evaluation'
diag_dir = results_dir / 'diagnostics'
reports_dir = eval_dir / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

fx_path = processed_dir / 'fx_aligned.csv'
exo_path = processed_dir / 'exogenous_php.csv'

fx = pd.read_csv(fx_path)
exo = pd.read_csv(exo_path)

fx['Date'] = pd.to_datetime(fx['Date'])
exo['Date'] = pd.to_datetime(exo['Date'])

df = fx.merge(exo, on='Date', how='left', suffixes=('', '_exo'))

target_ret = f'USD{active_target}_RET'
if target_ret not in df.columns:
    fallback = [c for c in df.columns if c.endswith('_RET')]
    target_ret = fallback[0] if fallback else None

print('Active target:', active_target)
print('Target return column:', target_ret)
print('Merged shape:', df.shape)
print('FX path:', fx_path)
print('Exogenous path:', exo_path)

Active target: PHP
Target return column: USDPHP_RET
Merged shape: (4230, 20)
FX path: data\processed\PHP\fx_aligned.csv
Exogenous path: data\processed\PHP\exogenous_php.csv


## 1) Parameter Persistence Check (from steps 03 and 04)

In [44]:
from scipy import stats

param_paths = []
for model_name in ['arima', 'arimax', 'var', 'varx']:
    p = results_dir / model_name / 'parameters.csv'
    if p.exists():
        param_paths.append((model_name, p))

for p in sorted(glob.glob(str(results_dir / 'hybrid_*' / 'parameters.csv'))):
    model_name = os.path.basename(os.path.dirname(p))
    param_paths.append((model_name, p))

if not param_paths:
    print('No parameter tables found.')
else:
    param_frames = []
    rename_map = {
        'Std..Error': 'Std. Error',
        't.value': 't-value',
        'P.value': 'P-value',
    }

    for model_name, p in param_paths:
        t = pd.read_csv(p).rename(columns=rename_map)
        for col in ['Estimate', 'Std. Error', 't-value', 'P-value']:
            if col in t.columns:
                t[col] = pd.to_numeric(t[col], errors='coerce')

        if {'Estimate', 'Std. Error'}.issubset(t.columns):
            can_fill_t = ('t-value' not in t.columns) or t['t-value'].isna()
            if np.isscalar(can_fill_t):
                can_fill_t = pd.Series([bool(can_fill_t)] * len(t), index=t.index)
            valid_t = can_fill_t & t['Estimate'].notna() & t['Std. Error'].notna() & (t['Std. Error'] > 0)
            if 't-value' not in t.columns:
                t['t-value'] = np.nan
            t.loc[valid_t, 't-value'] = t.loc[valid_t, 'Estimate'] / t.loc[valid_t, 'Std. Error']

            can_fill_p = ('P-value' not in t.columns) or t['P-value'].isna()
            if np.isscalar(can_fill_p):
                can_fill_p = pd.Series([bool(can_fill_p)] * len(t), index=t.index)
            valid_p = can_fill_p & t['t-value'].notna()
            if 'P-value' not in t.columns:
                t['P-value'] = np.nan
            t.loc[valid_p, 'P-value'] = 2 * stats.norm.sf(np.abs(t.loc[valid_p, 't-value']))

        t['ModelGroup'] = model_name
        param_frames.append(t)

    params_all = pd.concat(param_frames, ignore_index=True)
    display(params_all.head(25))

    needed = ['Parameter', 'Estimate', 'Std. Error', 't-value', 'P-value']
    completeness = pd.DataFrame({
        'field': needed,
        'available': [c in params_all.columns for c in needed],
        'non_null_pct': [float(params_all[c].notna().mean()) if c in params_all.columns else 0.0 for c in needed],
    })
    display(completeness)

    quality_rows = []
    for model_name, sub in params_all.groupby('ModelGroup'):
        quality_rows.append({
            'ModelGroup': model_name,
            'rows': len(sub),
            'Std.Error_non_null_pct': float(sub['Std. Error'].notna().mean()) if 'Std. Error' in sub.columns else 0.0,
            't_value_non_null_pct': float(sub['t-value'].notna().mean()) if 't-value' in sub.columns else 0.0,
            'P_value_non_null_pct': float(sub['P-value'].notna().mean()) if 'P-value' in sub.columns else 0.0,
        })
    quality_df = pd.DataFrame(quality_rows).sort_values('ModelGroup')
    display(quality_df)

    params_all.to_csv(reports_dir / f'checkin_parameters_snapshot_{active_target}.csv', index=False)
    quality_df.to_csv(reports_dir / f'checkin_parameters_quality_{active_target}.csv', index=False)
    print('Saved:', reports_dir / f'checkin_parameters_snapshot_{active_target}.csv')
    print('Saved:', reports_dir / f'checkin_parameters_quality_{active_target}.csv')

,Pair,Parameter,Estimate,Std. Error,t-value,P-value,ModelGroup,Equation
0,USDPHP_RET,ma1,-0.270735,0.016434,-16.474208,5.621972e-61,arima,NaN
1,CNYPHP_RET,ma1,-0.257674,0.016615,-15.508749,3.027216e-54,arima,NaN
2,JPYPHP_RET,ma1,-0.141939,0.017109,-8.296233,1.074644e-16,arima,NaN
3,HKDPHP_RET,ma1,-0.277461,0.016450,-16.866877,7.885733e-64,arima,NaN
4,SGDPHP_RET,ma1,-0.317244,0.015995,-19.833402,1.533123e-87,arima,NaN
5,USDPHP_RET,ma1,-0.315995,0.016204,-19.501098,1.074514e-84,arimax,NaN
6,USDPHP_RET,DXY_lr_lag1,0.194404,0.017568,11.065832,1.837495e-28,arimax,NaN
7,USDPHP_RET,PSEI_lr_lag1,-0.039959,0.006560,-6.091401,1.119271e-09,arimax,NaN
8,USDPHP_RET,VIX_diff_lag1,0.037962,0.004411,8.606034,7.563160e-18,arimax,NaN
9,USDPHP_RET,DFF_diff_lag1,0.262406,0.178162,1.472853,1.407908e-01,arimax,NaN


,field,available,non_null_pct
0,Parameter,True,1.0
1,Estimate,True,1.0
2,Std. Error,True,1.0
3,t-value,True,1.0
4,P-value,True,1.0


,ModelGroup,rows,Std.Error_non_null_pct,t_value_non_null_pct,P_value_non_null_pct
0,arima,5,1.0,1.0,1.0
1,arimax,35,1.0,1.0,1.0
2,hybrid_arima_mlp,20,1.0,1.0,1.0
3,hybrid_arima_svr,15,1.0,1.0,1.0
4,hybrid_arimax_mlp,20,1.0,1.0,1.0
5,hybrid_arimax_svr,15,1.0,1.0,1.0
6,hybrid_var_mlp,20,1.0,1.0,1.0
7,hybrid_var_svr,15,1.0,1.0,1.0
8,hybrid_varx_mlp,20,1.0,1.0,1.0
9,hybrid_varx_svr,15,1.0,1.0,1.0


Saved: results\PHP\evaluation\reports\checkin_parameters_snapshot_PHP.csv
Saved: results\PHP\evaluation\reports\checkin_parameters_quality_PHP.csv


In [45]:
def pick_col(patterns, columns):
    up = {c.upper(): c for c in columns}
    for pat in patterns:
        pat_u = pat.upper()
        for cu, c in up.items():
            if pat_u in cu:
                return c
    return None

coverage_spec = [
    ('Global Dollar Strength (DXY)', ['DXY']),
    ('Domestic Equity Sentiment (PSEi)', ['PSEI']),
    ('Energy-Import Pressure (Crude Oil)', ['CRUDE', 'OIL']),
    ('Risk Appetite (VIX)', ['VIX']),
    ('Regional Spillover (USD/CNY or USD/IDR proxy)', ['USDCNY', 'USDIDR', 'CNYPHP_RET', 'IDR']),
    ('US 10Y Yield (TNX)', ['TNX', 'US10Y', '10Y']),
    ('Remittance Seasonality Dummy', ['REMIT', 'DECEMBER_DUMMY', 'JUNE_DUMMY']),
    ('Forward Points / IRP Deviation', ['FORWARD', 'IRP']),
]

coverage_rows = []
for label, patterns in coverage_spec:
    col = pick_col(patterns, df.columns)
    coverage_rows.append({'Concept': label, 'DetectedColumn': col, 'Available': col is not None})

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

coverage_df.to_csv(reports_dir / f'exogenous_coverage_audit_{active_target}.csv', index=False)
print('Saved:', reports_dir / f'exogenous_coverage_audit_{active_target}.csv')

,Concept,DetectedColumn,Available
0,Global Dollar Strength (DXY),DXY_lr_lag1,True
1,Domestic Equity Sentiment (PSEi),PSEI_lr_lag1,True
2,Energy-Import Pressure (Crude Oil),CrudeOil_lr_lag1,True
3,Risk Appetite (VIX),VIX_diff_lag1,True
4,Regional Spillover (USD/CNY or USD/IDR proxy),CNYPHP_RET,True
5,US 10Y Yield (TNX),NaN,False
6,Remittance Seasonality Dummy,NaN,False
7,Forward Points / IRP Deviation,NaN,False


Saved: results\PHP\evaluation\reports\exogenous_coverage_audit_PHP.csv


In [49]:
work = df.copy()

bsp_diff_col = pick_col(['BSP_RRP_DIFF', 'BSP_DIFF', 'BSP_RRP_diff'], work.columns)
bsp_level_col = pick_col(['BSP_RRP', 'BSP_RATE'], work.columns)
if bsp_diff_col is not None:
    work['PolicyShock_BSP'] = work[bsp_diff_col]
elif bsp_level_col is not None:
    work['PolicyShock_BSP'] = work[bsp_level_col].diff()
else:
    work['PolicyShock_BSP'] = np.nan

regional_col = pick_col(['USDCNY_RET', 'USDCNY', 'USDIDR_RET', 'USDIDR', 'CNYPHP_RET'], work.columns)
if regional_col is not None:
    work['RegionalSpillover'] = work[regional_col]
else:
    work['RegionalSpillover'] = np.nan

tnx_col = pick_col(['TNX_DIFF', 'TNX', 'US10Y', '10Y'], work.columns)
if tnx_col is not None:
    work['US10YProxy'] = work[tnx_col]
else:
    work['US10YProxy'] = np.nan

work['Month'] = work['Date'].dt.month
work['Remit_December'] = (work['Month'] == 12).astype(int)
work['Remit_June'] = (work['Month'] == 6).astype(int)

forward_col = pick_col(['FORWARD_POINTS', 'FORWARD', 'FWD'], work.columns)
spot_col = pick_col([f'USD{active_target}'], work.columns)
if forward_col is not None and spot_col is not None:
    work['IRP_Deviation'] = work[forward_col] - work[spot_col]
else:
    work['IRP_Deviation'] = np.nan

engineered_cols = ['PolicyShock_BSP', 'RegionalSpillover', 'US10YProxy', 'Remit_December', 'Remit_June', 'IRP_Deviation']
display(work[['Date'] + engineered_cols].head(20))

,Date,PolicyShock_BSP,RegionalSpillover,US10YProxy,Remit_December,Remit_June,IRP_Deviation
0,2010-01-04,0.0,0.000000,NaN,0,0,NaN
1,2010-01-05,0.0,-0.012376,NaN,0,0,NaN
2,2010-01-06,0.0,-0.237403,NaN,0,0,NaN
3,2010-01-07,0.0,-0.047334,NaN,0,0,NaN
4,2010-01-08,0.0,-0.126901,NaN,0,0,NaN
5,2010-01-11,0.0,-0.677534,NaN,0,0,NaN
6,2010-01-12,0.0,-0.001462,NaN,0,0,NaN
7,2010-01-13,0.0,0.623541,NaN,0,0,NaN
8,2010-01-14,0.0,-0.189363,NaN,0,0,NaN
9,2010-01-15,0.0,0.427187,NaN,0,0,NaN


In [46]:
candidate_x = [
    c for c in [
        'DXY_lr_lag1',
        'PSEI_lr_lag1',
        'VIX_diff_lag1',
        'CrudeOil_lr_lag1',
        'DFF_diff_lag1',
        'PolicyShock_BSP',
        'RegionalSpillover',
        'US10YProxy',
        'IRP_Deviation',
    ] if c in work.columns
]

rows = []
for c in candidate_x:
    s = work[c].dropna().astype(float)
    if len(s) < 40:
        rows.append({'Variable': c, 'n': len(s), 'ADF_p': np.nan, 'KPSS_p': np.nan, 'Stationary_5pct': np.nan})
        continue
    try:
        adf_p = adfuller(s, autolag='AIC')[1]
    except Exception:
        adf_p = np.nan
    try:
        kpss_p = kpss(s, regression='c', nlags='auto')[1]
    except Exception:
        kpss_p = np.nan

    stationary = (adf_p < 0.05) and (kpss_p >= 0.05) if (not np.isnan(adf_p) and not np.isnan(kpss_p)) else np.nan
    rows.append({'Variable': c, 'n': len(s), 'ADF_p': adf_p, 'KPSS_p': kpss_p, 'Stationary_5pct': stationary})

stationarity_x = pd.DataFrame(rows).sort_values('Variable')
display(stationarity_x)
stationarity_x.to_csv(reports_dir / f'exogenous_stationarity_check_{active_target}.csv', index=False)
print('Saved:', reports_dir / f'exogenous_stationarity_check_{active_target}.csv')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4180\1781925929.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(s, regression='c', nlags='auto')[1]
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4180\1781925929.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(s, regression='c', nlags='auto')[1]
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4180\1781925929.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(s, regression='c', nlags='auto')[1]
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4180\1781925929.py:26: InterpolationWarning: The test statistic is outside of the range of p-value

,Variable,n,ADF_p,KPSS_p,Stationary_5pct
3,CrudeOil_lr_lag1,4229,1.911391e-22,0.1,True
4,DFF_diff_lag1,4229,4.411518e-12,0.1,True
0,DXY_lr_lag1,4229,0.000000e+00,0.1,True
8,IRP_Deviation,0,NaN,NaN,NaN
1,PSEI_lr_lag1,4229,0.000000e+00,0.1,True
5,PolicyShock_BSP,4229,0.000000e+00,0.1,True
6,RegionalSpillover,4230,4.723982e-21,0.1,True
7,US10YProxy,0,NaN,NaN,NaN
2,VIX_diff_lag1,4229,2.013467e-27,0.1,True


Saved: results\PHP\evaluation\reports\exogenous_stationarity_check_PHP.csv


In [47]:
vif_features = [c for c in ['DXY_lr_lag1', 'PSEI_lr_lag1', 'VIX_diff_lag1', 'CrudeOil_lr_lag1', 'DFF_diff_lag1', 'PolicyShock_BSP', 'RegionalSpillover', 'US10YProxy'] if c in work.columns]
vif_df = pd.DataFrame(columns=['Variable', 'VIF'])

if len(vif_features) >= 2:
    X0 = work[vif_features].copy()

    # Remove columns with too many missing values (e.g., fully missing TNX proxy).
    valid_cols = [c for c in X0.columns if X0[c].notna().mean() >= 0.8]
    dropped_missing = [c for c in X0.columns if c not in valid_cols]
    X1 = X0[valid_cols]

    # Remove (near) constant columns, which make VIF unstable.
    valid_var_cols = [c for c in X1.columns if X1[c].dropna().nunique() > 1]
    dropped_constant = [c for c in X1.columns if c not in valid_var_cols]
    X2 = X1[valid_var_cols]

    # Use median fill for remaining sparse points to avoid losing all rows.
    if len(X2.columns) >= 2:
        X = X2.apply(lambda s: s.fillna(s.median()), axis=0).astype(float)
        if len(X) > 50:
            X_const = sm.add_constant(X, has_constant='add')
            vifs = []
            for i, col in enumerate(X_const.columns):
                if col == 'const':
                    continue
                vifs.append({'Variable': col, 'VIF': variance_inflation_factor(X_const.values, i)})
            vif_df = pd.DataFrame(vifs).sort_values('VIF', ascending=False)

    print('VIF features input:', vif_features)
    print('Dropped (missing>=20%):', dropped_missing)
    print('Dropped (constant):', dropped_constant)
    print('Final VIF features:', list(vif_df['Variable']) if not vif_df.empty else [])

display(vif_df)
vif_df.to_csv(reports_dir / f'exogenous_vif_{active_target}.csv', index=False)
print('Saved:', reports_dir / f'exogenous_vif_{active_target}.csv')

VIF features input: ['DXY_lr_lag1', 'PSEI_lr_lag1', 'VIX_diff_lag1', 'CrudeOil_lr_lag1', 'DFF_diff_lag1', 'PolicyShock_BSP', 'RegionalSpillover', 'US10YProxy']
Dropped (missing>=20%): ['US10YProxy']
Dropped (constant): []
Final VIF features: ['VIX_diff_lag1', 'CrudeOil_lr_lag1', 'RegionalSpillover', 'DXY_lr_lag1', 'PSEI_lr_lag1', 'DFF_diff_lag1', 'PolicyShock_BSP']


,Variable,VIF
2,VIX_diff_lag1,1.071348
3,CrudeOil_lr_lag1,1.051694
6,RegionalSpillover,1.024352
0,DXY_lr_lag1,1.023787
1,PSEI_lr_lag1,1.013532
4,DFF_diff_lag1,1.008491
5,PolicyShock_BSP,1.000515


Saved: results\PHP\evaluation\reports\exogenous_vif_PHP.csv


In [50]:
def lag_corr(y, x, lag):
    if lag > 0:
        return y.corr(x.shift(lag))
    if lag < 0:
        return y.shift(-lag).corr(x)
    return y.corr(x)

lead_lag_rows = []
if target_ret is not None and target_ret in work.columns:
    y = work[target_ret].astype(float)
    scan_vars = [c for c in ['DXY_lr_lag1', 'PSEI_lr_lag1', 'VIX_diff_lag1', 'CrudeOil_lr_lag1', 'PolicyShock_BSP', 'RegionalSpillover', 'US10YProxy', 'IRP_Deviation'] if c in work.columns]
    for c in scan_vars:
        x = work[c].astype(float)
        lag_grid = list(range(-10, 11))
        corrs = [(lag, lag_corr(y, x, lag)) for lag in lag_grid]
        corrs = [(lag, val) for lag, val in corrs if pd.notna(val)]
        if not corrs:
            continue
        best_lag, best_corr = max(corrs, key=lambda z: abs(z[1]))
        lead_lag_rows.append({'Variable': c, 'BestLag': best_lag, 'BestCorr': best_corr})

lead_lag_df = pd.DataFrame(lead_lag_rows).sort_values('BestCorr', key=lambda s: s.abs(), ascending=False)
display(lead_lag_df)
lead_lag_df.to_csv(reports_dir / f'exogenous_lead_lag_scan_{active_target}.csv', index=False)
print('Saved:', reports_dir / f'exogenous_lead_lag_scan_{active_target}.csv')

,Variable,BestLag,BestCorr
5,RegionalSpillover,0,0.852989
0,DXY_lr_lag1,0,0.201843
2,VIX_diff_lag1,0,0.152942
1,PSEI_lr_lag1,-1,-0.122545
3,CrudeOil_lr_lag1,9,0.036425
4,PolicyShock_BSP,7,-0.032964


Saved: results\PHP\evaluation\reports\exogenous_lead_lag_scan_PHP.csv


In [51]:
break_path = diag_dir / 'structural_breaks_panel.csv'
break_effect_table = pd.DataFrame()

if target_ret is None or target_ret not in work.columns:
    print('Target return column unavailable for break-interaction model.')
elif not break_path.exists():
    print('No structural breaks file found at:', break_path)
else:
    breaks = pd.read_csv(break_path)
    date_col = 'break_date' if 'break_date' in breaks.columns else ('Break_Date' if 'Break_Date' in breaks.columns else None)
    pair_col = 'Pair' if 'Pair' in breaks.columns else None

    if date_col is None or pair_col is None:
        print('Break file missing required columns Pair/break_date.')
    else:
        dxy_col = pick_col(['DXY_LR_LAG1', 'DXY'], work.columns)
        if dxy_col is None:
            print('No DXY-type regressor found for slope-break test.')
        else:
            target_breaks = breaks[breaks[pair_col].astype(str).str.upper() == target_ret.upper()]
            if target_breaks.empty:
                target_breaks = breaks.copy()

            target_breaks[date_col] = pd.to_datetime(target_breaks[date_col], errors='coerce')
            if target_breaks[date_col].dropna().empty:
                print('No valid break dates available for interaction test.')
            else:
                bdate = target_breaks[date_col].dropna().min()
                tmp = work[['Date', target_ret, dxy_col]].dropna().copy()
                tmp['PostBreak'] = (tmp['Date'] >= bdate).astype(int)
                tmp['DXY_x_PostBreak'] = tmp[dxy_col] * tmp['PostBreak']

                y = tmp[target_ret].astype(float)
                X = sm.add_constant(tmp[[dxy_col, 'PostBreak', 'DXY_x_PostBreak']].astype(float), has_constant='add')
                fit = sm.OLS(y, X).fit(cov_type='HC3')

                coef = fit.params.rename('Estimate').to_frame()
                coef['Std. Error'] = fit.bse
                coef['t-value'] = fit.tvalues
                coef['P-value'] = fit.pvalues
                coef = coef.reset_index().rename(columns={'index': 'Parameter'})
                coef['BreakDate'] = bdate
                break_effect_table = coef
                display(break_effect_table)
                break_effect_table.to_csv(reports_dir / f'slope_break_interaction_{active_target}.csv', index=False)
                print('Saved:', reports_dir / f'slope_break_interaction_{active_target}.csv')

,Parameter,Estimate,Std. Error,t-value,P-value,BreakDate
0,const,-0.018648,0.019370,-0.962691,3.357029e-01,2013-01-22
1,DXY_lr_lag1,0.230102,0.039666,5.801034,6.590717e-09,2013-01-22
2,PostBreak,0.030358,0.020757,1.462561,1.435875e-01,2013-01-22
3,DXY_x_PostBreak,0.016858,0.045271,0.372375,7.096137e-01,2013-01-22


Saved: results\PHP\evaluation\reports\slope_break_interaction_PHP.csv


In [48]:
upgrade_rows = []

for _, r in coverage_df.iterrows():
    if not bool(r['Available']):
        upgrade_rows.append({'Priority': 'High', 'Item': r['Concept'], 'Status': 'Missing', 'Recommendation': 'Add data source and transform to stationary form.'})

if not vif_df.empty:
    high_vif = vif_df[vif_df['VIF'] > 10]
    for _, r in high_vif.iterrows():
        upgrade_rows.append({'Priority': 'High', 'Item': f"VIF>10: {r['Variable']}", 'Status': 'Risk', 'Recommendation': 'Drop/orthogonalize correlated regressors and re-estimate.'})

if 'stationarity_x' in globals() and not stationarity_x.empty:
    non_stationary = stationarity_x[stationarity_x['Stationary_5pct'] == False]
    for _, r in non_stationary.iterrows():
        upgrade_rows.append({'Priority': 'High', 'Item': f"Non-stationary X: {r['Variable']}", 'Status': 'Risk', 'Recommendation': 'Difference/log-difference this regressor before modeling returns.'})

if 'lead_lag_df' in globals() and not lead_lag_df.empty:
    for _, r in lead_lag_df.iterrows():
        upgrade_rows.append({'Priority': 'Medium', 'Item': f"Lag refinement for {r['Variable']}", 'Status': 'Info', 'Recommendation': f"Test model with lag {int(r['BestLag'])} (highest abs corr={r['BestCorr']:.3f})."})

upgrade_df = pd.DataFrame(upgrade_rows)
display(upgrade_df.head(50))

upgrade_df.to_csv(reports_dir / f'checkin_upgrade_checklist_{active_target}.csv', index=False)
print('Saved:', reports_dir / f'checkin_upgrade_checklist_{active_target}.csv')

,Priority,Item,Status,Recommendation
0,High,US 10Y Yield (TNX),Missing,Add data source and transform to stationary form.
1,High,Remittance Seasonality Dummy,Missing,Add data source and transform to stationary form.
2,High,Forward Points / IRP Deviation,Missing,Add data source and transform to stationary form.
3,Medium,Lag refinement for RegionalSpillover,Info,Test model with lag 0 (highest abs corr=0.853).
4,Medium,Lag refinement for DXY_lr_lag1,Info,Test model with lag 0 (highest abs corr=0.202).
5,Medium,Lag refinement for VIX_diff_lag1,Info,Test model with lag 0 (highest abs corr=0.153).
6,Medium,Lag refinement for PSEI_lr_lag1,Info,Test model with lag -1 (highest abs corr=-0.123).
7,Medium,Lag refinement for CrudeOil_lr_lag1,Info,Test model with lag 9 (highest abs corr=0.036).
8,Medium,Lag refinement for PolicyShock_BSP,Info,Test model with lag 7 (highest abs corr=-0.033).


Saved: results\PHP\evaluation\reports\checkin_upgrade_checklist_PHP.csv
